# nb03 — Evaluate ReliefForge (Kaggle, GPU or CPU)

Compares zero-shot vs Stage A vs Stage B on the held-out test scenes, including the
resolution study S ∈ {518, 756, 1036} (Day-4 gate). Requires the dataset (Input) and
the exports from nb02 (upload `train_a/export` and `train_b/export` as a Kaggle Dataset
named `reliefforge-exports`, or re-attach the nb02 output).

In [ ]:
%%bash
cd /kaggle/working
rm -rf reliefforge
git clone --depth 1 https://github.com/<YOUR_USERNAME>/reliefforge.git reliefforge
pip -q install PyYAML mlflow
ls /kaggle/input && ls /kaggle/input/reliefforge-exports 2>/dev/null || echo "attach the exports dataset"

In [ ]:
%%bash
cd /kaggle/working/reliefforge
DATA=/kaggle/input/reliefforge-dataset
EXP=/kaggle/input/reliefforge-exports
python src/evaluate.py --config configs/config.yaml --data $DATA --zero-shot \
  --out artifacts/eval --sizes 518 756 1036 2>&1 | tail -6
python src/evaluate.py --config configs/config.yaml --data $DATA --model $EXP/train_a/export \
  --label stage_a --out artifacts/eval_a --sizes 518 2>&1 | tail -4
python src/evaluate.py --config configs/config.yaml --data $DATA --model $EXP/train_b/export \
  --label stage_b --out artifacts/eval_b --sizes 518 756 1036 2>&1 | tail -6

In [ ]:
import pandas as pd
frames = []
import glob
for f in glob.glob('/kaggle/working/reliefforge/artifacts/eval*/summary.csv'):
    frames.append(pd.read_csv(f))
summary = pd.concat(frames).sort_values(['model', 'S'])
summary

In [ ]:
# qualitative figure: input | GT | zero-shot | fine-tuned
%%bash
cd /kaggle/working/reliefforge
python - <<'EOF'
import sys
sys.path.insert(0, 'src')
import torch
from ml.evaluate import load_model, qualitative_figure
root = '/kaggle/input/reliefforge-dataset'
models = {
    'zeroshot': load_model('depth-anything/Depth-Anything-V2-Small-hf', 'cuda' if torch.cuda.is_available() else 'cpu'),
    'stage_b': load_model('/kaggle/input/reliefforge-exports/train_b/export', 'cuda' if torch.cuda.is_available() else 'cpu'),
}
out = qualitative_figure(models, root, 'artifacts/figures/qualitative.png', n=8,
                         device='cuda' if torch.cuda.is_available() else 'cpu')
print(out)
EOF

In [ ]:
!cd /kaggle/working/reliefforge && tar -czf /kaggle/working/reliefforge_eval.tar.gz artifacts mlruns
!ls -lh /kaggle/working/reliefforge_eval.tar.gz
# Download, unpack into the repo: artifacts/eval*/csv files, figures, mlruns/.